# Cymbal Foods: write the text corpus

**Pipeline step 11.** Notebook 10 built the Cymbal Foods dataset: orders, supply, inventory and the rest. This notebook writes the text that goes with it: the words a customer, a quality analyst or an account rep would have written. Gemini writes every document, and each one is written only from a block of facts taken from the dataset, so the text and the tables always agree.

| Type | Documents | Written by | What the labs use it for |
| :-- | --: | :-- | :-- |
| Product descriptions | 835 | the website team | semantic search ("something smoky that isn't hot") |
| Reviews | 70,000 | customers, about orders they really placed | finding complaint themes with AI functions, and tracing a bad review back to the batch and the supplier lot |
| Recipes | 3,000 | the test kitchen | linking products to dishes |
| Supplier quality reports | 15,000 | quality analysts, about real lots | the failed paprika lot, found from its retest report |
| Account notes | 10,000 | account reps, about real restaurant accounts | private notes an agent may only show to the account's own rep |

Every document is embedded with `gemini-embedding-001` (3,072 dimensions), so the labs can search by meaning.

| | |
| :-- | :-- |
| **Runs in** | Colab Enterprise in a lab project with Vertex AI |
| **Reads** | the dataset at `gs://class-demo/alloydb-labs/cymbalfoods-alloydb-workshop/dataset/v1/`, and the code from the workshop's GitHub repo |
| **Writes** | only under that dataset prefix: `corpus/` while it works, then the corpus tables and the re-cut slices |
| **Calls** | `gemini-3.8-flash` (global endpoint) to write, `gemini-embedding-001` (regional) to embed |

### The two runs

| Run | Set in section 1 | What happens | Cost and time |
| :-- | :-- | :-- | :-- |
| **1** | `RUN_MODE = "sample"` (the default) | 25 documents of each type, checked, shown for review, and a cost estimate for the full run worked out from what the sample really used | a few cents, a few minutes |
| **2** | `RUN_MODE = "full"` | every document, checked and repaired, then embedded, then the corpus tables are written and the slices re-cut | about $100–150 and 2–4 hours (run 1 gives the real numbers) |

**Run all** is always safe. Run 2 saves its progress to the bucket as it goes, so if the runtime stops, Run all again and it carries on where it left off. The last cell prints one box to copy and paste back.

The rules for the text and the checks that enforce them live in `datagen/cfgen/corpus.py`; the calls to Gemini are in `datagen/cfgen/gemini.py`.

---
## 1 · Configuration

The run flags are at the top. Everything else is a setting with a reason next to it.

In [ ]:
# ===== RUN FLAGS: the only things you normally change ========================================
RUN_MODE     = "sample"   # "sample" = 25 of each type for review (run 1); "full" = every document (run 2)
TYPES        = "all"      # "all", or a comma-separated subset: product_descriptions, reviews, recipes,
                          #   supplier_quality_reports, account_notes
RERUN_SAMPLE = False      # True = write a fresh sample even though one is saved for this prompt version
STAGE        = True       # full run: write the corpus tables to the dataset and re-cut the slices
# =============================================================================================
REPO_URL   = "https://github.com/haggman/cymbalfoods-alloydb-workshop.git"
BRANCH     = "main"
GCS_ROOT   = "gs://class-demo/alloydb-labs/cymbalfoods-alloydb-workshop"
DATASET    = f"{GCS_ROOT}/dataset/v1"
NEVER_WRITE = "gs://class-demo/alloydb-labs/cymbalgoal/"    # another team's labs read this; never written

GEN_MODEL       = "gemini-3.8-flash"
GEN_LOCATIONS   = ["global"]                                # Gemini 3.x is served only from the global endpoint
EMBED_MODEL     = "gemini-embedding-001"
EMBED_DIM       = 3072                                       # the labs embed queries in SQL at the model's full width
EMBED_LOCATIONS = ["us-central1", "us-east1", "europe-west4"]
SEED            = 2026
SAMPLE_WORKERS  = 8
MAX_WORKERS, MAX_WORKERS_CEILING = 50, 150                   # the throttle grows the pool while nothing is rate-limited
EMBED_WORKERS   = 32
EMBED_SHARD     = 2000          # embeddings per shard file in the bucket (the resume unit)
SAVE_EVERY      = 200           # documents between local checkpoints
PUSH_EVERY_S    = 120           # seconds between checkpoint uploads
REPAIR_ROUNDS   = 3
REPAIR_LIMIT    = 0.06          # more hard failures than this share of a type is a prompt problem: stop and look

In [ ]:
import datetime as dt, html, io, json, math, os, random, re, subprocess, sys, time
from collections import Counter, defaultdict
from pathlib import Path

RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
WORK = Path("/content/cymbalfoods-corpus") if Path("/content").exists() else Path.home() / "cymbalfoods-corpus"
WORK.mkdir(parents=True, exist_ok=True)
REPO, PQ, LOCAL = WORK / "repo", WORK / "out" / "parquet", WORK / "corpus"
for d in (PQ, LOCAL):
    d.mkdir(parents=True, exist_ok=True)
FAKE = bool(os.environ.get("CF_FAKE_GEMINI"))     # offline test hook; unset in Colab
HB = {}

def hb(section, line):
    HB.setdefault(section, []).append(str(line))
    print(line)

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode:
        raise RuntimeError(f"{cmd}\n{r.stderr[-2000:]}")
    return r

if RUN_MODE not in ("sample", "full"):
    raise ValueError('RUN_MODE must be "sample" or "full"')
hb("Run", f"run {RUN_ID}  mode={RUN_MODE}  types={TYPES}  rerun_sample={RERUN_SAMPLE}  stage={STAGE}" + ("  [FAKE GEMINI]" if FAKE else ""))

---
## 2 · Get the code

A shallow clone of the workshop repo, the generator's libraries, and the Gemini SDK (`google-genai`, recent enough for Gemini 3's `thinking_level`).

In [ ]:
S = "2 · Code"
if REPO.exists():
    sh(f"git -C {REPO} fetch -q --depth 1 origin {BRANCH} && git -C {REPO} reset -q --hard origin/{BRANCH}")
else:
    sh(f"git clone -q --depth 1 --branch {BRANCH} {REPO_URL} {REPO}")
DATAGEN = REPO / "datagen"
sh(f"{sys.executable} -m pip install -q -r {DATAGEN / 'requirements.txt'}")
if not FAKE:      # upgrade only the SDK; --upgrade on the requirements line would also move Colab's pandas
    sh(f"{sys.executable} -m pip install -q --upgrade google-genai")
sys.path.insert(0, str(DATAGEN))
import numpy as np, pandas as pd
from cfgen import corpus as K, gemini as G, writer, schemas
commit = sh(f"git -C {REPO} rev-parse --short HEAD").stdout.strip()
TYPE_LIST = K.TYPES if TYPES == "all" else [t.strip() for t in TYPES.split(",")]
bad = [t for t in TYPE_LIST if t not in K.TYPES]
if bad:
    raise ValueError(f"unknown types {bad}; choose from {K.TYPES}")
PV = K.PROMPT_VERSION
CORPUS = f"{DATASET}/corpus"
try:
    import google.genai as _g
    _gv = getattr(_g, "__version__", "?")
except ImportError:
    _gv = "not installed"
hb(S, f"repo {REPO_URL} @ {commit}; prompt version {PV}; types {', '.join(TYPE_LIST)}")
hb(S, f"python {sys.version.split()[0]}  pandas {pd.__version__}  google-genai {_gv}  runtime {os.cpu_count()} vCPU")

---
## 3 · The bucket guard

Every Cloud Storage call goes through `gcs()`. Writes are allowed **only** under the dataset prefix, and anything under `alloydb-labs/cymbalgoal/` is refused, because another team's labs read it. Nothing is ever deleted except the slice folders this notebook re-cuts, which are synchronized exactly. The probe writes one tiny object to prove this identity can write.

In [ ]:
S = "3 · Bucket"
GCS_LOG = []
_WRITE_VERBS = {"cp", "rsync"}

def _under(uri, prefix):
    p = prefix.rstrip("/")
    return uri == p or uri.startswith(p + "/")

def gcs(*args, check=True):
    args = list(args)
    verb = args[0]
    uris = [a for a in args[1:] if a.startswith("gs://")]
    if verb not in ("cp", "rsync", "ls", "du", "cat"):
        raise PermissionError(f"gcs(): '{verb}' refused")
    if any(_under(u, NEVER_WRITE) for u in uris):
        raise PermissionError(f"gcs(): refusing anything under {NEVER_WRITE}")
    if any(not _under(u, DATASET) for u in uris):
        raise PermissionError(f"gcs(): outside {DATASET}: {uris}")
    GCS_LOG.append(" ".join(["gcloud storage"] + args))
    r = subprocess.run(["gcloud", "storage"] + args, capture_output=True, text=True)
    if check and r.returncode:
        raise RuntimeError(f"gcloud storage {' '.join(args)}\n{r.stderr[-1500:]}")
    return r

def exists(uri):
    """True / False, or None when the listing itself failed (so a failed restore is never mistaken for a first run)."""
    r = gcs("ls", uri, check=False)
    if r.returncode == 0:
        return True
    err = r.stderr.lower()
    return False if ("matched no objects" in err or "not found" in err or "no urls matched" in err) else None

def put(local, uri):
    gcs("cp", str(local), uri)

def get(uri, local):
    Path(local).parent.mkdir(parents=True, exist_ok=True)
    return gcs("cp", uri, str(local), check=False).returncode == 0

PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT") or sh("gcloud config get-value project 2>/dev/null").stdout.strip()
ACCOUNT = sh("gcloud config get-value account 2>/dev/null").stdout.strip()
probe = WORK / "probe.txt"
probe.write_text(f"write probe {RUN_ID}\n")
WRITE_OK = gcs("cp", str(probe), f"{DATASET}/_runs/{RUN_ID}-corpus-probe.txt", check=False).returncode == 0
hb(S, f"project {PROJECT}  account {ACCOUNT}  write to {DATASET}: {'OK' if WRITE_OK else 'FAILED'}")
if not WRITE_OK:
    print(f"""
This identity can't write to {DATASET}/. From an account that owns class-demo, run:

  gcloud storage buckets add-iam-policy-binding gs://class-demo \\
    --member="user:{ACCOUNT}" --role=roles/storage.objectAdmin \\
    --condition='expression=resource.name.startsWith("projects/_/buckets/class-demo/objects/alloydb-labs/cymbalfoods-alloydb-workshop/"),title=cymbalfoods-workshop-prefix'

then Run all again.""")

---
## 4 · Preflight

Finds a location that serves each model, with one tiny call to each (a fraction of a cent). Nothing that costs money runs unless this passes and the bucket probe passed.

In [ ]:
S = "4 · Preflight"
GEN_LOCATION = EMBED_LOCATION = None
PREFLIGHT_ERRORS = []

def _probe_gen(loc, cfg):
    last = None
    for attempt in range(3):          # a fresh project sometimes answers its first calls with 429 or 503
        try:
            # keep the client referenced for the whole call: a google-genai Client that is garbage-collected
            # closes its connection, which fails the request with "the client has been closed"
            client = G.client_factory(PROJECT, loc)()
            return client.models.generate_content(
                model=GEN_MODEL, contents='Reply with the JSON object {"description": "ready"}', config=cfg), None
        except Exception as exc:  # noqa: BLE001
            last = exc
            if G.status_code(exc) not in (429, 500, 503):
                break
            time.sleep(10 * (attempt + 1))
    return None, last

for loc in GEN_LOCATIONS:
    # the configuration the run uses: JSON schema, thinking level LOW, no automatic function calling
    r, exc = _probe_gen(loc, G.gen_config("Reply in JSON.", K.SCHEMA["product_descriptions"], 0, None, max_output_tokens=1024))
    if r is not None:
        GEN_LOCATION = loc
        print(f"✅ {GEN_MODEL} @ {loc}: {(r.text or '').strip()[:40]!r}")
        break
    msg = f"{GEN_MODEL} @ {loc}: {type(exc).__name__}: {' '.join(str(exc).split())[:400]}"
    print("   " + msg)
    PREFLIGHT_ERRORS.append(msg)
    if not FAKE:                       # a plain call tells a configuration problem from an access problem
        from google.genai import types as _gt
        r2, exc2 = _probe_gen(loc, _gt.GenerateContentConfig(max_output_tokens=256))
        note = ("plain call (no schema, no thinking config) WORKS" if r2 is not None else
                f"plain call also fails: {type(exc2).__name__}: {' '.join(str(exc2).split())[:300]}")
        print("   " + note)
        PREFLIGHT_ERRORS.append(f"   {loc}: {note}")
for loc in EMBED_LOCATIONS:
    try:
        _get = G.client_factory(PROJECT, loc)
        _k, v, err = G.embed_one(("probe", "ready"), G.Meter(), get_client=_get,
                                 model=EMBED_MODEL, dim=EMBED_DIM)
        if v is not None:
            EMBED_LOCATION = loc
            print(f"✅ {EMBED_MODEL} @ {loc}: {len(v)} dims")
            break
        print(f"   {EMBED_MODEL} @ {loc}: {err[:160]}")
        PREFLIGHT_ERRORS.append(f"{EMBED_MODEL} @ {loc}: {err[:300]}")
    except Exception as exc:  # noqa: BLE001
        print(f"   {EMBED_MODEL} @ {loc}: {type(exc).__name__}: {str(exc)[:160]}")
        PREFLIGHT_ERRORS.append(f"{EMBED_MODEL} @ {loc}: {type(exc).__name__}: {str(exc)[:300]}")
PREFLIGHT_OK = bool(WRITE_OK and GEN_LOCATION and EMBED_LOCATION)
GEN_CLIENT = G.client_factory(PROJECT, GEN_LOCATION) if GEN_LOCATION else None
EMB_CLIENT = G.client_factory(PROJECT, EMBED_LOCATION) if EMBED_LOCATION else None
hb(S, f"gen {GEN_MODEL}@{GEN_LOCATION}  embed {EMBED_MODEL}@{EMBED_LOCATION}  " + ("PASSED" if PREFLIGHT_OK else "FAILED: later sections skip"))
if not PREFLIGHT_OK:
    for e in PREFLIGHT_ERRORS:
        hb(S, "  " + e)

---
## 5 · The facts

Notebook 10 staged the tables and, under `parquet/_facts/`, one fact table per document type: which reviews exist and what each must say, which lots get which reports, and so on. This section downloads the tables it needs and turns each fact row into the **fact block** Gemini writes from. One example of each is printed below.

The fact files' fingerprint names this corpus: a corpus is only valid for the facts it was written from, and section 11 refuses to stage it against different ones.

In [ ]:
S = "5 · Facts"
t0 = time.time()
need = K.TABLES + ["orders", "_facts"]
for t in need:
    if not (PQ / t).exists():
        gcs("rsync", "--recursive", f"{DATASET}/parquet/{t}", str(PQ / t))
FACTS_FP = K.fingerprint(PQ)
T = K.load(PQ)
JOBS = {t: j for t, j in K.build_jobs(T).items() if t in TYPE_LIST}
JD = {t: {r["job_id"]: r for r in j.to_dict("records")} for t, j in JOBS.items()}
hb(S, f"facts fingerprint {FACTS_FP}; " + ", ".join(f"{t} {len(j):,}" for t, j in JOBS.items()) +
      f"; {sum(len(j) for j in JOBS.values()):,} documents; built in {time.time() - t0:,.0f} s")
for t, j in JOBS.items():
    print(f"\n——— {t}: one fact block ———\n{j.facts.iloc[min(3, len(j) - 1)]}")

---
## 6 · The rules

Each type has its own instruction (below), and every instruction ends with the same rules: write only from the facts; never invent numbers, dates, prices or people; never name a real company, brand, store or carrier (only Cymbal Foods); make no health or nutrition claims; don't write "it's not X, it's Y"; never mention the facts or the prompt; plain prose, no markdown.

Reviews go 10 to a request, which cuts the input cost of the instruction. Every response is structured JSON.

**The checks** (in `corpus.qa`). The same checks run on the sample, the full run and every repair. A **hard** failure is regenerated with a note saying what was wrong; a **soft** flag is only counted.

| | Check |
| :-- | :-- |
| hard, all types | missing or unparseable response, or an empty field; length outside the hard range; a real brand or retailer; a health or nutrition claim; "it's not X, it's Y"; prompt plumbing ("the facts", "JSON"); markdown, brackets or garbled characters |
| hard, all but recipes | a number over 10 that isn't in the facts |
| hard, by type | **descriptions:** a mild product called fiery or very hot, a hot one with no mention of heat, a smoked one not called smoky. **Reviews:** every planted theme comes through (a musty review says musty), and no musty wording in a review without that theme; a title of 1–12 words. **Recipes:** every Cymbal Foods product named exactly in the ingredients; at least three numbered steps. **Quality reports:** the lot code; a fail stated as a fail; the paprika retest names the musty complaints, and no other report mentions mustiness. **Notes:** the musty complaints say musty, substitutions say what was substituted, preferences are recorded in the chef's terms |
| soft | length outside the target range; a review that mentions its stars; a description that never uses the product's name |

In [ ]:
for t in TYPE_LIST:
    print(f"——— {t} (temperature {K.TEMPERATURE[t]}) ———\n{K.SYSTEM[t]}\n")

---
## 7 · Run 1: the sample

Runs with `RUN_MODE = "sample"`. 25 documents of each type, chosen to cover the planted signals and the cases most likely to go wrong: every review theme (three musty ones), the paprika lot's two reports, the musty complaints and the Saigon substitutions, the cinnamon and chile preferences, the smoked and the very hot products, gift sets, blends and pantry items.

Each document is checked; hard failures get one repair, as in the full run. The sample is saved to the bucket, so a later Run all shows it again without paying twice (`RERUN_SAMPLE = True` writes a new one).

Below the documents: what each type cost per document, and what the full run will cost at that rate.

In [ ]:
S = "7 · Sample"
SAMPLE = {}
CAL = {}

def _gen_for(t, reqs, workers, ceiling, label):
    cfg = G.gen_config(K.SYSTEM[t], K.SCHEMA[t], K.TEMPERATURE[t], SEED)
    rcfg = lambda: G.gen_config(K.SYSTEM[t], K.SCHEMA[t], K.TEMPERATURE[t], random.randint(1, 2**31 - 1))
    got = []
    m = G.run_pool(reqs, lambda r, mt: G.generate_one(r, get_client=GEN_CLIENT, model=GEN_MODEL, config=cfg,
                                                      parse=lambda txt, ids: K.parse(t, txt, ids), meter=mt, repair_config=rcfg),
                   workers=workers, ceiling=ceiling, label=label, on_result=got.append)
    return got, m

def _per_doc(res):
    """Split a request's tokens over its documents (reviews come 10 to a request)."""
    out = {}
    for r in res:
        n = len(r["job_ids"])
        for j in r["job_ids"]:
            o = (r["outputs"] or {}).get(j)
            out[j] = dict(out=o, error=r["error"] if not o else "", in_tok=r["in_tok"] / n, out_tok=r["out_tok"] / n,
                          think_tok=r["think_tok"] / n, attempts=r["attempts"], request_id=r["request_id"])
    return out

if RUN_MODE != "sample":
    hb(S, "skipped (RUN_MODE is full)")
elif not PREFLIGHT_OK:
    hb(S, "skipped: preflight failed")
else:
    for t in TYPE_LIST:
        uri = f"{CORPUS}/samples/{PV}/{t}.jsonl"
        local = LOCAL / "samples" / PV / f"{t}.jsonl"
        if not RERUN_SAMPLE and exists(uri) and get(uri, local):
            SAMPLE[t] = [json.loads(l) for l in local.read_text().splitlines() if l.strip()]
            print(f"{t}: reusing the saved sample ({len(SAMPLE[t])})")
            continue
        ids = K.pick_sample(t, JOBS[t])
        sub = JOBS[t][JOBS[t].job_id.isin(ids)]
        t0 = time.time()
        res, m = _gen_for(t, K.requests(t, sub), SAMPLE_WORKERS, SAMPLE_WORKERS, f"sample {t}")
        D = _per_doc(res)
        first = {j: K.qa(t, JD[t][j], D[j]["out"]) for j in ids}
        need = {j: K.repair_note(t, h) for j, (h, _s) in first.items() if h}
        if need:
            rreq = [dict(r, repair=True) for r in K.requests(t, sub[sub.job_id.isin(need)], repair=need)]
            rres, m2 = _gen_for(t, rreq, SAMPLE_WORKERS, SAMPLE_WORKERS, f"repair {t}")
            for j, v in _per_doc(rres).items():
                if v["out"]:
                    D[j].update(out=v["out"], error="")
                D[j]["repair_in_tok"], D[j]["repair_out_tok"] = v["in_tok"], v["out_tok"]
        rows = []
        for j in ids:
            h, s_ = K.qa(t, JD[t][j], D[j]["out"])
            rows.append(dict(job_id=j, facts=JD[t][j]["facts"], **(D[j]["out"] or {}), error=D[j]["error"],
                             first_draft_hard=first[j][0], qa_hard=h, qa_soft=s_,
                             in_tok=D[j]["in_tok"], out_tok=D[j]["out_tok"], think_tok=D[j]["think_tok"],
                             repair_in_tok=D[j].get("repair_in_tok", 0), repair_out_tok=D[j].get("repair_out_tok", 0),
                             seconds=round((time.time() - t0) / max(len(ids), 1), 2)))
        SAMPLE[t] = rows
        local.parent.mkdir(parents=True, exist_ok=True)
        local.write_text("\n".join(json.dumps(x, ensure_ascii=False, default=str) for x in rows) + "\n")
        put(local, uri)
    # ---- calibration: what each type cost per document, projected onto the full run
    total = 0.0
    for t, rows in SAMPLE.items():
        n = max(len(rows), 1)
        tin = sum(r["in_tok"] for r in rows) / n
        tout = sum(r["out_tok"] for r in rows) / n
        tth = sum(r["think_tok"] for r in rows) / n
        rep = sum(1 for r in rows if r["first_draft_hard"]) / n
        per = (tin * G.PRICE_IN_PER_M + tout * G.PRICE_OUT_PER_M) / 1e6 * (1 + rep)
        chars = sum(len(K.embed_text(t, r)) for r in rows if not r["error"] and all(f in r for f in K.FIELDS[t])) / n
        emb = chars / 4 * G.PRICE_EMBED_PER_M / 1e6
        full_n = len(JOBS[t])
        CAL[t] = dict(tin=tin, tout=tout, tth=tth, repair_share=rep, per_doc=per, full=per * full_n, emb=emb * full_n, n=full_n)
        total += CAL[t]["full"] + CAL[t]["emb"]
        words = [K._words(K.embed_text(t, r)) for r in rows if all(f in r for f in K.FIELDS[t])]
        hard_first = Counter(x.split(":")[0] for r in rows for x in r["first_draft_hard"])
        hard_left = Counter(x.split(":")[0] for r in rows for x in r["qa_hard"])
        soft = Counter(x.split(":")[0] for r in rows for x in r["qa_soft"])
        hb(S, f"{t}: {len(rows)} docs; first drafts with hard failures {sum(1 for r in rows if r['first_draft_hard'])} "
              f"{dict(hard_first)}; still failing after one repair {sum(1 for r in rows if r['qa_hard'])} {dict(hard_left)}; "
              f"soft {dict(soft)}; words median {int(np.median(words)) if words else 0} ({min(words) if words else 0}–{max(words) if words else 0})")
        hb(S, f"   tokens/doc in {tin:,.0f} out {tout:,.0f} (reasoning {tth:,.0f}); ${per * 1000:,.2f} per 1,000 docs "
              f"→ full {full_n:,} docs ≈ ${CAL[t]['full']:,.2f} + embeddings ≈ ${CAL[t]['emb']:,.2f}")
    if CAL:
        hb(S, f"PROJECTED FULL RUN ≈ ${total:,.0f} at online prices (types run: {', '.join(CAL)})")

### Read the sample

Every sample document with the facts it was written from. The tags show what the checks found: **red** is a hard failure, **amber** a soft flag, and *repaired* means the first draft failed and this is the second.

In [ ]:
from IPython.display import HTML, display

def _doc_html(t, r):
    tags = "".join(f'<span style="background:#c0392b;color:#fff;padding:1px 6px;border-radius:3px;margin-right:4px">{html.escape(x)}</span>' for x in r.get("qa_hard", []))
    tags += "".join(f'<span style="background:#d68910;color:#fff;padding:1px 6px;border-radius:3px;margin-right:4px">{html.escape(x)}</span>' for x in r.get("qa_soft", []))
    if r.get("first_draft_hard"):
        tags += '<span style="background:#7d8c99;color:#fff;padding:1px 6px;border-radius:3px">repaired</span>'
    body = "".join(f"<p><b>{html.escape(f)}</b><br>{html.escape(str(r.get(f) or r.get('error') or '')).replace(chr(10), '<br>')}</p>" for f in K.FIELDS[t])
    return (f'<div style="border:1px solid #888;border-radius:6px;padding:8px 12px;margin:8px 0">{tags}{body}'
            f'<details><summary>facts</summary><pre style="white-space:pre-wrap">{html.escape(r["facts"])}</pre></details></div>')

if SAMPLE:
    for t, rows in SAMPLE.items():
        display(HTML(f"<h3>{t} ({len(rows)})</h3>" + "".join(_doc_html(t, r) for r in rows)))
else:
    print("no sample in this run")

---
## 8 · Run 2: generate everything

Runs with `RUN_MODE = "full"`, one type at a time. The pool refills a slot the moment a call returns, and the throttle grows it while nothing is rate-limited.

**Checkpoints.** Every few hundred documents the results so far are saved on the runtime's disk, and every two minutes that checkpoint is copied to `corpus/checkpoints/` in the bucket. A failed call stays in the checkpoint with its error, so the next run retries exactly those.

**Resuming.** If the runtime stops, Run all again: the checkpoint is restored and only the missing or failed documents are generated. If the bucket listing *errors* (rather than finding nothing), the section refuses to start, because a failed restore would otherwise look exactly like a first run.

In [ ]:
S = "8 · Generate"
GEN = {t: {} for t in TYPE_LIST}
CKPT_STATUS = {}

def ckpt_paths(t):
    return LOCAL / "checkpoints" / PV / f"{t}.parquet", f"{CORPUS}/checkpoints/{PV}/{t}.parquet"

def save_ckpt(t, push):
    lp, uri = ckpt_paths(t)
    lp.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([dict(job_id=j, out=json.dumps(v["out"]) if v["out"] else None, error=v["error"], in_tok=v["in_tok"],
                       out_tok=v["out_tok"], think_tok=v["think_tok"], attempts=v["attempts"], request_id=v["request_id"])
                  for j, v in GEN[t].items()]).to_parquet(lp, index=False)
    if push:
        put(lp, uri)

def restore_ckpt(t):
    lp, uri = ckpt_paths(t)
    e = exists(uri)
    if e is None:
        return "ERROR"
    rows = pd.read_parquet(lp) if lp.exists() else None
    if e:
        tmp = lp.with_suffix(".bucket.parquet")
        get(uri, tmp)
        b = pd.read_parquet(tmp)
        if rows is None or len(b) > len(rows):
            rows = b
    if rows is not None:
        GEN[t] = {r["job_id"]: dict(out=json.loads(r["out"]) if r["out"] else None, error=r["error"] or "", in_tok=r["in_tok"],
                                   out_tok=r["out_tok"], think_tok=r["think_tok"], attempts=r["attempts"], request_id=r["request_id"])
                  for r in rows.to_dict("records")}
    return f"{len(GEN[t]):,} restored" if GEN[t] else "none"

def run_type(t, reqs, label):
    cfg = G.gen_config(K.SYSTEM[t], K.SCHEMA[t], K.TEMPERATURE[t], SEED)
    rcfg = lambda: G.gen_config(K.SYSTEM[t], K.SCHEMA[t], K.TEMPERATURE[t], random.randint(1, 2**31 - 1))
    state = {"last_push": time.time()}

    def on_result(r):
        n = len(r["job_ids"])
        for j in r["job_ids"]:
            o = (r["outputs"] or {}).get(j)
            GEN[t][j] = dict(out=o, error=r["error"] if not o else "", in_tok=r["in_tok"] / n, out_tok=r["out_tok"] / n,
                             think_tok=r["think_tok"] / n, attempts=r["attempts"], request_id=r["request_id"])

    def on_save():
        push = time.time() - state["last_push"] >= PUSH_EVERY_S
        save_ckpt(t, push)
        if push:
            state["last_push"] = time.time()
    per_req = max(1, len(reqs[0]["job_ids"])) if reqs else 1
    m = G.run_pool(reqs, lambda r, mt: G.generate_one(r, get_client=GEN_CLIENT, model=GEN_MODEL, config=cfg,
                                                      parse=lambda txt, ids: K.parse(t, txt, ids), meter=mt, repair_config=rcfg),
                   workers=MAX_WORKERS, ceiling=MAX_WORKERS_CEILING, label=label, on_result=on_result,
                   save_every=max(1, SAVE_EVERY // per_req), on_save=on_save)
    save_ckpt(t, True)
    return m

GEN_COMPLETE = {}
if RUN_MODE != "full":
    hb(S, "skipped (RUN_MODE is sample)")
elif not PREFLIGHT_OK:
    hb(S, "skipped: preflight failed")
else:
    for t in TYPE_LIST:
        CKPT_STATUS[t] = restore_ckpt(t)
        if CKPT_STATUS[t] == "ERROR":
            hb(S, f"🔴 {t}: REFUSING TO START, the checkpoint listing failed. Fix bucket access and Run all again.")
            continue
        todo = JOBS[t][~JOBS[t].job_id.map(lambda j: j in GEN[t] and GEN[t][j]["out"] is not None)]
        print(f"{t}: checkpoint {CKPT_STATUS[t]}; to generate {len(todo):,}")
        t0 = time.time()
        m = run_type(t, K.requests(t, todo), f"generate {t}") if len(todo) else None
        ok = sum(1 for v in GEN[t].values() if v["out"])
        GEN_COMPLETE[t] = ok == len(JOBS[t])
        hb(S, f"{t}: {ok:,} of {len(JOBS[t]):,} generated (checkpoint {CKPT_STATUS[t]})" +
              (f"; this run {len(todo):,} in {(time.time() - t0) / 60:,.1f} min, ≈${m.cost():,.2f}, 429s {m.rate_limited}, failures {m.failures}" if m else ""))

---
## 9 · Checks and repair

Runs once a type is fully generated. The checks from section 6 run over every document; each hard failure is regenerated with a note saying what was wrong, up to three rounds. If more than 6% of a type fails, it stops instead: a pattern that large is a prompt problem, not bad luck, and the handback lists the reasons. The checked results are saved to `corpus/results/`.

In [ ]:
S = "9 · Checks"
RES = {}
QA_CLEAN = {}
if RUN_MODE == "full" and PREFLIGHT_OK:
    for t in TYPE_LIST:
        if not GEN_COMPLETE.get(t):
            hb(S, f"{t}: waiting for generation to finish")
            continue
        QA = {j: K.qa(t, JD[t][j], GEN[t][j]["out"]) for j in JD[t]}
        repaired = set()
        for rnd in range(1, REPAIR_ROUNDS + 1):
            need = {j: K.repair_note(t, h) for j, (h, _s) in QA.items() if h}
            why = Counter(x.split(":")[0] for j in need for x in QA[j][0])
            hb(S, f"{t}: pass {rnd}: {len(need):,} hard failures {dict(why.most_common(8))}")
            if rnd == 1:
                ex = defaultdict(list)
                for j in need:
                    for x in QA[j][0]:
                        k = x.split(":")[0]
                        if len(ex[k]) < 2:
                            ex[k].append(f"{x[:90]} :: {' | '.join(str(v)[:160].replace(chr(10), ' / ') for v in (GEN[t][j]['out'] or {}).values())}")
                for k, v in ex.items():
                    hb(S, f"   e.g. {k}: " + " || ".join(v))
            if not need:
                break
            if len(need) > REPAIR_LIMIT * len(JD[t]):
                hb(S, f"🔴 {t}: {len(need):,} hard failures is over {REPAIR_LIMIT:.0%}; not repairing. Read the reasons first.")
                break
            sub = JOBS[t][JOBS[t].job_id.isin(need)]
            run_type(t, [dict(r, repair=True) for r in K.requests(t, sub, repair=need)], f"repair {t} {rnd}")
            repaired |= set(need)
            QA.update({j: K.qa(t, JD[t][j], GEN[t][j]["out"]) for j in need})
        rows = [dict(job_id=j, out=json.dumps(GEN[t][j]["out"]) if GEN[t][j]["out"] else None, in_tok=GEN[t][j]["in_tok"],
                     out_tok=GEN[t][j]["out_tok"], think_tok=GEN[t][j]["think_tok"], repaired=j in repaired,
                     qa_hard="; ".join(QA[j][0]), qa_soft="; ".join(QA[j][1])) for j in JD[t]]
        RES[t] = pd.DataFrame(rows)
        lp = LOCAL / "results" / PV / f"{t}.parquet"
        lp.parent.mkdir(parents=True, exist_ok=True)
        RES[t].to_parquet(lp, index=False)
        put(lp, f"{CORPUS}/results/{PV}/{t}.parquet")
        left = RES[t][RES[t].qa_hard != ""]
        QA_CLEAN[t] = len(left) == 0
        soft = Counter(r.split(":")[0] for x in RES[t].qa_soft for r in x.split("; ") if r)
        cost = (RES[t].in_tok.sum() * G.PRICE_IN_PER_M + RES[t].out_tok.sum() * G.PRICE_OUT_PER_M) / 1e6
        words = RES[t].out.map(lambda o: K._words(K.embed_text(t, json.loads(o))) if o else 0)
        hb(S, f"{t}: {len(RES[t]):,} docs; repaired {len(repaired):,}; still failing {len(left):,}; soft {dict(soft.most_common(6))}; "
              f"words median {words.median():.0f} (p5 {words.quantile(.05):.0f}, p95 {words.quantile(.95):.0f}); kept docs ≈${cost:,.2f}")
        for _, x in left.head(5).iterrows():
            hb(S, f"   still failing {x.job_id}: {x.qa_hard}")
        for _, x in RES[t][RES[t].out.notna()].sample(n=min(3, len(RES[t])), random_state=7).iterrows():
            o = json.loads(x.out)
            hb(S, f"   random {x.job_id}: " + " | ".join(str(v).replace(chr(10), " / ") for v in o.values()))
else:
    hb(S, "skipped (sample run)" if RUN_MODE == "sample" else "skipped: preflight failed")

---
## 10 · Embeddings

`gemini-embedding-001` at 3,072 dimensions, task type `RETRIEVAL_DOCUMENT`, of exactly the text each table stores in the embedded columns (a review's title and body; a recipe's title, ingredients and method). Shards of 2,000 go to `corpus/emb/` as they finish, so a stopped runtime loses at most one shard; a rerun restores the finished ones and carries on. A type is embedded only once it passes its checks.

In [ ]:
S = "10 · Embeddings"
EMB = {t: {} for t in TYPE_LIST}
EMB_DONE = {}
if RUN_MODE == "full" and PREFLIGHT_OK:
    for t in TYPE_LIST:
        if not QA_CLEAN.get(t):
            hb(S, f"{t}: waiting for clean checks")
            continue
        prefix = f"{CORPUS}/emb/{PV}/{t}"
        ldir = LOCAL / "emb" / PV / t
        ldir.mkdir(parents=True, exist_ok=True)
        if exists(prefix):
            gcs("rsync", "--recursive", prefix, str(ldir))
        for f in sorted(ldir.glob("shard-*.parquet")):
            d = pd.read_parquet(f)
            EMB[t].update({k: np.asarray(v, dtype=np.float32) for k, v in zip(d.job_id, d.embedding)})
        texts = {x.job_id: K.embed_text(t, json.loads(x.out)) for x in RES[t].itertuples()}
        todo = [(j, s) for j, s in texts.items() if j not in EMB[t]]
        shard_no = len(list(ldir.glob("shard-*.parquet")))
        t0, errors, em = time.time(), [], G.Meter()
        print(f"{t}: {len(EMB[t]):,} restored, {len(todo):,} to embed")
        for start in range(0, len(todo), EMBED_SHARD):
            got = []
            def on(r):
                k, v, err = r
                (got.append((k, v)) if v is not None else errors.append((k, err)))
            G.run_pool(todo[start:start + EMBED_SHARD],
                       lambda it, mt: G.embed_one(it, mt, get_client=EMB_CLIENT, model=EMBED_MODEL, dim=EMBED_DIM),
                       workers=EMBED_WORKERS, ceiling=EMBED_WORKERS * 2, label=f"embed {t}", on_result=on, meter=em, unit="doc")
            if got:
                sp = ldir / f"shard-{shard_no:04d}.parquet"
                pd.DataFrame({"job_id": [g[0] for g in got], "embedding": [g[1] for g in got]}).to_parquet(sp, index=False)
                put(sp, f"{prefix}/{sp.name}")
                shard_no += 1
                EMB[t].update(dict(got))
        EMB_DONE[t] = len(EMB[t]) == len(RES[t])
        hb(S, f"{t}: {len(EMB[t]):,} of {len(RES[t]):,} embedded in {(time.time() - t0) / 60:,.1f} min; errors {len(errors)}"
              + (f" (first: {errors[0][1][:120]})" if errors else ""))
else:
    hb(S, "skipped (sample run)" if RUN_MODE == "sample" else "skipped: preflight failed")

---
## 11 · Write the corpus tables and re-cut the slices

Runs when every type is generated, checked and embedded (all five, in one or several runs). It assembles the six corpus tables (`recipe_products` comes with `recipes`), writes them to `parquet/` beside the relational tables, then re-cuts every slice in `datagen/slices.yaml` with the corpus included and stages the result. Each slice with vectors also gets an `indexes-vector.sql` with its ScaNN indexes sized to the rows it loads.

In [ ]:
S = "11 · Stage"
STAGED = False
ready = RUN_MODE == "full" and all(EMB_DONE.get(t) for t in K.TYPES)
if RUN_MODE != "full":
    hb(S, "skipped (sample run)")
elif not ready:
    hb(S, "waiting: every type must be generated, checked and embedded (" +
          ", ".join(f"{t} {'ready' if EMB_DONE.get(t) else 'not yet'}" for t in K.TYPES) + ")")
elif not STAGE:
    hb(S, "skipped (STAGE = False)")
elif K.fingerprint(PQ) != FACTS_FP:
    hb(S, "🔴 the facts changed during the run; not staging")
else:
    t0 = time.time()
    OUT = WORK / "out"
    for t in K.TYPES:
        outs = {x.job_id: json.loads(x.out) for x in RES[t].itertuples()}
        for name, df in K.assemble(t, T, outs, EMB[t]).items():
            if (PQ / name).exists():
                for f in (PQ / name).glob("*.parquet"):
                    f.unlink()
            rows, nbytes = writer.write_parquet(df, name, PQ / name / "part-000.parquet")
            gcs("rsync", "--recursive", "--delete-unmatched-destination-objects", str(PQ / name), f"{DATASET}/parquet/{name}")
            hb(S, f"  {name}: {rows:,} rows, {nbytes / 1e6:,.1f} MB Parquet")
    meta = dict(prompt_version=PV, facts_fingerprint=FACTS_FP, model=GEN_MODEL, embedding=f"{EMBED_MODEL}@{EMBED_DIM}",
                written=RUN_ID, counts={t: len(RES[t]) for t in K.TYPES})
    (LOCAL / "corpus-manifest.json").write_text(json.dumps(meta, indent=1))
    put(LOCAL / "corpus-manifest.json", f"{CORPUS}/manifest.json")
    # the relational tables too, then re-cut every slice with the corpus included
    for d in ("parquet", "sql"):
        gcs("rsync", "--recursive", f"{DATASET}/{d}", str(OUT / d))
    get(f"{DATASET}/manifest.json", OUT / "manifest.json")
    r = subprocess.run([sys.executable, str(DATAGEN / "generate_dataset.py"), "--out", str(OUT), "--cut-only"], capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode not in (0, 3):
        raise RuntimeError(f"re-cutting the slices failed:\n{r.stderr[-3000:]}")
    M = json.loads((OUT / "manifest.json").read_text())
    for name in M["slices"]:
        gcs("rsync", "--recursive", "--delete-unmatched-destination-objects", str(OUT / "slices" / name), f"{DATASET}/slices/{name}")
        sm = json.loads((OUT / "slices" / name / "manifest.json").read_text())
        corp = {k: v["rows"] for k, v in sm["tables"].items() if v["has_vectors"]}
        hb(S, f"  slice {name}: {sm['totals']['rows']:,} rows ({sm['totals']['vectors']:,} with vectors {corp}), "
              f"{sm['totals']['bytes'] / 1e6:,.0f} MB; schema {sm['schema_files']}; missing groups {sm.get('groups_missing')}")
    gcs("cp", str(OUT / "manifest.json"), f"{DATASET}/manifest.json")
    STAGED = True
    hb(S, f"staged in {(time.time() - t0) / 60:,.1f} min")

---
## 12 · Where things stand

In [ ]:
if not PREFLIGHT_OK:
    NEXT = "Preflight failed: see sections 3 and 4, fix, and Run all again."
elif RUN_MODE == "sample":
    NEXT = "Sample written. Read it (section 7) and paste the handback back. When it's approved, set RUN_MODE = \"full\" and Run all."
elif STAGED:
    NEXT = "DONE: the corpus is written, embedded and staged, and every slice is re-cut."
elif any(s == "ERROR" for s in CKPT_STATUS.values()):
    NEXT = "Stopped: a checkpoint listing failed. Fix bucket access and Run all again."
elif not all(GEN_COMPLETE.get(t) for t in TYPE_LIST):
    NEXT = "Generation not finished (some calls failed). Run all again to retry them."
elif not all(QA_CLEAN.get(t) for t in TYPE_LIST):
    NEXT = "Some documents still fail the checks. Paste the handback back before running again."
elif not all(EMB_DONE.get(t) for t in TYPE_LIST):
    NEXT = "Embeddings not finished. Run all again to carry on."
else:
    NEXT = "Types done; run the remaining types (TYPES = \"all\") to stage."
hb("Status", NEXT)

---
## 13 · Handback

One box with everything from this run, and a **Copy** button. If your browser blocks the button, the text is selected for you; copy it with **Edit → Copy**. The same text is saved to the bucket under `dataset/v1/_runs/`. After a sample run the box includes every sample document with its facts, so it is long.

In [ ]:
from IPython.display import HTML, display
lines = ["=" * 72, f"CYMBAL FOODS CORPUS: 11 HANDBACK  run {RUN_ID}  mode {RUN_MODE}  prompt {PV}", "=" * 72]
for sec, ls_ in HB.items():
    lines += ["", f"## {sec}", *ls_]
if SAMPLE:
    lines += ["", "## Sample documents"]
    for t, rows in SAMPLE.items():
        lines += ["", f"### {t}"]
        for r in rows:
            flags = (" HARD " + "; ".join(r["qa_hard"]) if r["qa_hard"] else "") + (" soft " + "; ".join(r["qa_soft"]) if r["qa_soft"] else "") + \
                    (" (repaired: " + "; ".join(r["first_draft_hard"]) + ")" if r["first_draft_hard"] else "")
            lines.append(f"- [{r['job_id'][:8]}]{flags}")
            lines.append("  FACTS " + " | ".join(r["facts"].splitlines()))
            for f in K.FIELDS[t]:
                lines.append(f"  {f.upper()} " + str(r.get(f) or r.get("error") or "").replace("\n", " / "))
writes = [x for x in GCS_LOG if x.split()[2] in ("cp", "rsync") and "gs://" in x.split()[-1]]
lines += ["", "## Bucket audit", f"{len(GCS_LOG)} calls, {len(writes)} writes, all under {DATASET}/"
          if all(x.split()[-1].startswith(DATASET) for x in writes) else "WRITE OUTSIDE THE DATASET PREFIX: " + " | ".join(writes)]
text = "\n".join(lines)
(WORK / "handback.txt").write_text(text)
if WRITE_OK:
    gcs("cp", str(WORK / "handback.txt"), f"{DATASET}/_runs/{RUN_ID}-corpus-handback.txt", check=False)
display(HTML(f"""
<button onclick="navigator.clipboard.writeText(document.getElementById('hb11').innerText).then(()=>this.innerText='Copied')"
  style="padding:6px 14px;margin-bottom:6px;cursor:pointer">Copy</button>
<pre id="hb11" style="white-space:pre-wrap;border:1px solid #888;padding:10px;max-height:520px;overflow:auto">{html.escape(text)}</pre>
<script>var r=document.createRange();r.selectNodeContents(document.getElementById('hb11'));var s=window.getSelection();s.removeAllRanges();s.addRange(r);</script>
"""))